# AML 3303 — Week 3 In-Class Lab (30 minutes)
### pandas + NumPy only

## Scenario

**NorthLine Electronics** is an online store. The ops team exported two tables for you:
`customers` and `orders`. They want a quick read on which product categories actually
make money, and which city orders the most.

The export is realistic, which means it is slightly messy: a missing quantity, a missing
price, a duplicate order, and one order from a customer who is not in the customer list.

Fill in every cell marked `# TODO`. The hints sit directly above each one.

## 0. Setup

Run this cell once. Do not change it.

In [1]:
# Run this cell first. It creates the two DataFrames you will work with.
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["Aarav Shah","Bianca Rossi","Chen Wei","Diya Patel",
             "Emeka Obi","Farah Haddad","Grace Kim","Hugo Martin"],
    "city": ["Ottawa","Toronto","Ottawa","Montreal",
             "Toronto","Calgary","Montreal","Ottawa"],
})

orders = pd.DataFrame({
    "order_id":   [101,102,103,104,105,106,107,108,109,110,
                   111,112,113,114,115,116,117,118,119,120,118,121],
    "customer_id":[  1,  2,  1,  3,  4,  5,  2,  6,  3,  7,
                     8,  5,  1,  4,  6,  7,  2,  8,  3,  5,  8,  9],
    "order_date": ["2024-05-02","2024-05-02","2024-05-04","2024-05-05","2024-05-07",
                   "2024-05-08","2024-05-09","2024-05-11","2024-05-12","2024-05-13",
                   "2024-05-15","2024-05-16","2024-05-18","2024-05-19","2024-05-20",
                   "2024-05-22","2024-05-23","2024-05-25","2024-05-26","2024-05-28",
                   "2024-05-25","2024-05-29"],
    "category":   ["Laptops","Headphones","Headphones","Monitors","Laptops",
                   "Keyboards","Monitors","Laptops","Keyboards","Headphones",
                   "Monitors","Laptops","Keyboards","Headphones","Monitors",
                   "Laptops","Keyboards","Headphones","Laptops","Monitors",
                   "Headphones","Keyboards"],
    "quantity":   [1,2,1,2,1,3,1,1,np.nan,1,4,1,2,1,1,1,1,2,1,2,2,1],
    "unit_price": [1299.99,89.50,89.50,219.00,1499.00,45.00,219.00,1099.00,45.00,np.nan,
                   219.00,1299.99,45.00,89.50,219.00,1499.00,45.00,89.50,1299.99,219.00,
                   89.50,45.00],
    "status":     ["delivered","delivered","returned","delivered","cancelled",
                   "delivered","delivered","delivered","delivered","delivered",
                   "delivered","returned","delivered","delivered","delivered",
                   "delivered","cancelled","delivered","delivered","delivered",
                   "delivered","delivered"],
})

print("orders:", orders.shape, " customers:", customers.shape)
orders.head()

orders: (22, 7)  customers: (8, 3)


,order_id,customer_id,order_date,category,quantity,unit_price,status
0,101,1,2024-05-02,Laptops,1.0,1299.99,delivered
1,102,2,2024-05-02,Headphones,2.0,89.50,delivered
2,103,1,2024-05-04,Headphones,1.0,89.50,returned
3,104,3,2024-05-05,Monitors,2.0,219.00,delivered
4,105,4,2024-05-07,Laptops,1.0,1499.00,cancelled


## 1. First look (3 min)

**Hints**
- `df.shape` → rows and columns
- `df.info()` → column types and non-null counts
- `df.describe()` → quick numeric summary
- `df["category"].value_counts()` → how many orders per category

In [10]:
# TODO: print the shape, run info(), and show how many orders there are per category
orders.shape
orders.info()
orders['category'].value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     22 non-null     int64  
 1   customer_id  22 non-null     int64  
 2   order_date   22 non-null     object 
 3   category     22 non-null     object 
 4   quantity     21 non-null     float64
 5   unit_price   21 non-null     float64
 6   status       22 non-null     object 
dtypes: float64(2), int64(2), object(3)
memory usage: 1.3+ KB


category
Laptops       6
Headphones    6
Monitors      5
Keyboards     5
Name: count, dtype: int64

## 2. A first (wrong) answer (5 min)

Compute revenue per category **before** cleaning anything, counting delivered orders only.
Keep this result: you will compare against it at the end.

**Hints**
- Filter rows: `orders[orders["status"] == "delivered"]`
- New column: `df["revenue"] = df["quantity"] * df["unit_price"]`
- Group: `df.groupby("category")["revenue"].sum().sort_values(ascending=False)`

In [20]:
# TODO: revenue by category using the RAW orders (delivered only)
delivered_raw = orders.loc[orders["status"] == "delivered"].copy()
delivered_raw["revenue"] = delivered_raw["quantity"] * delivered_raw["unit_price"]
revenue_raw = delivered_raw.groupby("category")["revenue"].sum().sort_values(ascending=False)
revenue_raw

category
Laptops       5197.98
Monitors      2190.00
Headphones     626.50
Keyboards      270.00
Name: revenue, dtype: float64

## 3. Find the problems (5 min)

**Hints**
- Duplicates: `df.duplicated().sum()`, and `df[df.duplicated(keep=False)]` to see them
- Missing values: `df.isna().sum()`
- Orphan rows: `~orders["customer_id"].isin(customers["customer_id"])`

In [22]:
# 1. Count duplicate rows, then show both copies of the duplicate.
print("Fully duplicated rows:", orders.duplicated().sum())
print(orders.loc[orders.duplicated(keep=False)].to_string(index=False))

# 2. Count missing values in every column.
print("\nMissing values per column:")
print(orders.isna().sum())

# 3. Identify orders whose customer_id is absent from customers.
orphan_orders = orders.loc[~orders["customer_id"].isin(customers["customer_id"])]
print("\nOrders with an unknown customer:")
print(orphan_orders.to_string(index=False))


Fully duplicated rows: 1
 order_id  customer_id order_date   category  quantity  unit_price    status  revenue
      118            8 2024-05-25 Headphones       2.0        89.5 delivered    179.0
      118            8 2024-05-25 Headphones       2.0        89.5 delivered    179.0

Missing values per column:
order_id       0
customer_id    0
order_date     0
category       0
quantity       1
unit_price     1
status         0
revenue        2
dtype: int64

Orders with an unknown customer:
 order_id  customer_id order_date  category  quantity  unit_price    status  revenue
      121            9 2024-05-29 Keyboards       1.0        45.0 delivered     45.0


## 4. Clean it (7 min)

**Hints**
- `orders_clean = orders.drop_duplicates()`
- Fill a price with the median for that category:
  `df.groupby("category")["unit_price"].transform("median")`
- `df["quantity"].fillna(1)` — one unit is a reasonable default for this shop
- `pd.to_datetime(df["order_date"])`
- Make quantity an integer at the end: `.astype(int)`

In [23]:
orders_clean = orders.copy()

# 1. Drop fully duplicated rows; copy to allow safe column assignments.
orders_clean = orders_clean.drop_duplicates().copy()

# 2. Fill each missing price with the median for that product category.
category_median_price = orders_clean.groupby("category")["unit_price"].transform("median")
orders_clean["unit_price"] = orders_clean["unit_price"].fillna(category_median_price)

# 3. Use one unit for the missing quantity, then store quantities as integers.
orders_clean["quantity"] = orders_clean["quantity"].fillna(1).astype(int)

# 4. Parse dates so they can be used for time-based analysis.
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"])

orders_clean.info()


<class 'pandas.core.frame.DataFrame'>
Index: 21 entries, 0 to 21
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     21 non-null     int64         
 1   customer_id  21 non-null     int64         
 2   order_date   21 non-null     datetime64[ns]
 3   category     21 non-null     object        
 4   quantity     21 non-null     int64         
 5   unit_price   21 non-null     float64       
 6   status       21 non-null     object        
 7   revenue      19 non-null     float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(2)
memory usage: 1.5+ KB


## 5. NumPy: revenue column (3 min)

Cancelled and returned orders earn **no** revenue.

**Hints**
- `np.where(condition, value_if_true, value_if_false)`
- Condition: `orders_clean["status"] == "delivered"`
- `np.round(x, 2)` if you want tidy numbers

In [24]:
# Delivered orders earn quantity * unit_price; other statuses earn zero.
orders_clean["revenue"] = np.round(
    np.where(
        orders_clean["status"] == "delivered",
        orders_clean["quantity"] * orders_clean["unit_price"],
        0.0,
    ),
    2,
)

print(f"Total revenue: ${orders_clean['revenue'].sum():,.2f}")
# The mean includes all cleaned orders, including the zero-revenue orders.
print(f"Mean order value (all {len(orders_clean)} orders): ${orders_clean['revenue'].mean():,.2f}")
print(f"Largest single order revenue: ${orders_clean['revenue'].max():,.2f}")



Total revenue: $8,239.98
Mean order value (all 21 orders): $392.38
Largest single order revenue: $1,499.00


## 6. Merge and answer the business question (5 min)

**Hints**
- `orders_clean.merge(customers, on="customer_id", how="left")`
- A left merge keeps the order from the unknown customer, with `city` as NaN. That is
  usually what you want: do not silently delete money.
- `.groupby("city")["revenue"].sum().sort_values(ascending=False)`

In [25]:
# A left merge preserves every order, including the unknown customer's order.
merged = orders_clean.merge(customers, on="customer_id", how="left")

# dropna=False includes the unknown city so its revenue is still counted.
revenue_by_city = (
    merged.groupby("city", dropna=False)["revenue"]
    .sum()
    .sort_values(ascending=False)
    .round(2)
)
revenue_by_city


city
Ottawa      4227.98
Montreal    1678.00
Calgary     1318.00
Toronto      971.00
NaN           45.00
Name: revenue, dtype: float64

## 7. Compare with step 2 (2 min)

**Hint:** `pd.DataFrame({"raw": revenue_raw, "clean": revenue_clean})` lines the two up side by side.

In [27]:
# Cancelled and returned orders already have zero revenue.
revenue_clean = (
    orders_clean.groupby("category")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .round(2)
)

comparison = pd.DataFrame({"raw": revenue_raw, "clean": revenue_clean})
comparison["change"] = (comparison["clean"] - comparison["raw"]).round(2)
comparison = comparison.sort_values("clean", ascending=False)
comparison


,raw,clean,change
category,,,
Laptops,5197.98,5197.98,0.0
Monitors,2190.00,2190.00,0.0
Headphones,626.50,537.00,-89.5
Keyboards,270.00,315.00,45.0


**Question:** two categories changed. Which ones, and what caused each change?

*Your answer:*
- Headphones: revenue decreased from $626.50 to $537.00, a net decrease of $89.50. Removing the duplicate order 118 subtracted $179.00, while filling the missing price on delivered order 110 with the Headphones median of $89.50 added $89.50.
- Keyboards: revenue increased from $270.00 to $315.00, an increase of $45.00. Filling the missing quantity on delivered order 109 with 1 added 1 x $45.00 in revenue.
Laptops and Monitors did not change. In the raw calculation, pandas ignores missing revenue values when summing, so the two incomplete orders contributed no revenue until their missing values were filled.



## 8. Wrap up (optional, if time)

Write two sentences for the NorthLine ops team: which category to push, and one data
quality issue they should fix at the source.

*Your answer:*
Prioritize Laptops for sales promotion because they generated the most delivered-order revenue ($5,197.98), though costs would be needed to confirm profitability. Add a uniqueness check on order_id at export to prevent duplicate orders such as 118 from inflating revenue.

---

### Submission
Save the notebook, push it to your GitHub repo, and paste the repo link in the class channel.

### Stretch goals
1. Which customer spent the most? (merge, then group by name)
2. Add a `week` column from `order_date` and show revenue per week.
   Hint: `df["order_date"].dt.isocalendar().week`
3. Ask Claude or Copilot to write the cleaning step for you, then compare it with yours.
   Did it make the same choices about the missing quantity? Was it right?

Optional stretch goal 1: top customer
Keep the unknown customer in the report as its own NaN group. The highest-spending identified customer is Chen Wei, with $1,782.99 in delivered-order revenue.

In [30]:
revenue_by_customer = (
    merged.groupby(["customer_id", "name"], dropna=False)["revenue"]
    .sum()
    .sort_values(ascending=False)
    .round(2)
)
revenue_by_customer


customer_id  name        
3            Chen Wei        1782.99
7            Grace Kim       1588.50
1            Aarav Shah      1389.99
6            Farah Haddad    1318.00
8            Hugo Martin     1055.00
5            Emeka Obi        573.00
2            Bianca Rossi     398.00
4            Diya Patel        89.50
9            NaN               45.00
Name: revenue, dtype: float64

##Optional stretch goal 2: weekly revenue
Use ISO week numbers from the cleaned dates. All orders are from 2024, so grouping by week alone works for this dataset; data spanning multiple years would also need an ISO year column.

In [31]:
orders_clean["week"] = orders_clean["order_date"].dt.isocalendar().week
revenue_by_week = orders_clean.groupby("week")["revenue"].sum().round(2)
revenue_by_week


week
18    1916.99
19    1498.00
20    1145.00
21    3196.99
22     483.00
Name: revenue, dtype: float64